# ZGLMM368 — Reservas

**Tabela:** `dev_procurement.corp_curated.tbl_ds_pro_zglmm368`
**Transação SAP:** ZGLMM368 · **Colunas:** 36
**Clustering declarado:** `num_reserva`, `cod_material`

---

## Objetivo
Mapear o comportamento desta tabela **antes** de qualquer comparação com o SAP.
O resultado alimenta a base de conhecimento do agente de validação e define o cenário de teste.

## Como usar
1. Execute a célula **1** para criar os widgets, depois ajuste os filtros no topo (opcional).
2. Execute a célula **2** — ela cria a view `base`, usada por todas as demais.
3. Execute as células na ordem e leia a coluna **`veredito`** de cada resultado.
4. Exporte o notebook executado para a pasta de conhecimento do agente.

## Aviso metodológico
Contagem de linhas **não** é evidência de qualidade. Erros de colapso de granularidade
preservam o total. Ver seções **4**, **5** e **14**.


## 1. Widgets de recorte

Execute uma vez. Deixe vazio para analisar a base completa.

In [0]:
-- 1. WIDGETS (deixe vazio = sem filtro)
-- Parametros criados via notebook UI (SQL warehouse nao suporta CREATE WIDGET).
-- Referenciados nas demais celulas via :f_cod_centro, :f_cod_empresa_compensacao, etc.
SELECT
  :f_cod_centro AS filtro_centro,
  :f_cod_empresa_compensacao AS filtro_empresa_compensacao,
  :f_cod_tipo_movimento AS filtro_tipo_movimento,
  :f_st_reserva AS filtro_st_reserva;

## 2. View `base`

Aplica os filtros dos widgets uma única vez. **Todas** as células seguintes consultam `base`.

In [0]:
-- 2. VIEW BASE (aplica os filtros dos widgets)
CREATE OR REPLACE TEMP VIEW base AS
SELECT * FROM dev_procurement.corp_curated.tbl_ds_pro_zglmm368
WHERE (:f_cod_centro = '' OR `cod_centro` = :f_cod_centro)
  AND (:f_cod_empresa_compensacao = '' OR `cod_empresa_compensacao` = :f_cod_empresa_compensacao)
  AND (:f_cod_tipo_movimento = '' OR `cod_tipo_movimento` = :f_cod_tipo_movimento)
  AND (:f_st_reserva = '' OR `st_reserva` = :f_st_reserva);

SELECT COUNT(*) AS linhas_na_base FROM base;

## 3. Metadados e histórico de carga

Formato, particionamento, clustering real e última atualização.
Divergência entre clustering declarado e chave real é o primeiro indício de problema.

In [0]:
DESCRIBE EXTENDED dev_procurement.corp_curated.tbl_ds_pro_zglmm368;

In [0]:
-- Formato, tamanho e particoes (falha se nao for Delta)
DESCRIBE DETAIL dev_procurement.corp_curated.tbl_ds_pro_zglmm368;

In [0]:
-- Ultimas operacoes de escrita (falha se for view ou nao-Delta)
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm368 LIMIT 20;

## 4. Granularidade real

`linhas ÷ chaves distintas`. Razão maior que 1,00 significa que existe uma dimensão
adicional multiplicando as linhas — é preciso descobrir **qual** (seção 14).

In [0]:
-- 4. GRANULARIDADE REAL: linhas / chaves distintas
-- Razao > 1,00 significa que existe dimensao adicional multiplicando linhas.
WITH t AS (SELECT COUNT(*) AS total FROM base),
g AS (
SELECT 'num_reserva' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_reserva` FROM base)
UNION ALL
SELECT 'num_reserva + cod_material' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_reserva`, `cod_material` FROM base)
UNION ALL
SELECT 'num_reserva + num_item_reserva' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM base)
UNION ALL
SELECT 'num_reserva + num_item_reserva + cod_material' AS chave, COUNT(*) AS combinacoes_distintas
  FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `cod_material` FROM base)
)
SELECT g.chave, t.total AS linhas, g.combinacoes_distintas,
       ROUND(t.total / g.combinacoes_distintas, 4) AS linhas_por_chave,
       CASE WHEN g.combinacoes_distintas = t.total THEN 'CHAVE UNICA'
            ELSE 'NAO UNICA - ha dimensao adicional' END AS veredito
FROM g CROSS JOIN t
ORDER BY linhas_por_chave;

## 5. Duplicidade por chave candidata

Quantas combinações se repetem e qual o pior caso.

In [0]:
-- 5. DUPLICIDADE POR CHAVE
SELECT 'num_reserva' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_reserva`, COUNT(*) AS qtd FROM base GROUP BY `num_reserva` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'num_reserva + cod_material' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_reserva`, `cod_material`, COUNT(*) AS qtd FROM base GROUP BY `num_reserva`, `cod_material` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'num_reserva + num_item_reserva' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_reserva`, `num_item_reserva`, COUNT(*) AS qtd FROM base GROUP BY `num_reserva`, `num_item_reserva` HAVING COUNT(*) > 1)
UNION ALL
SELECT 'num_reserva + num_item_reserva + cod_material' AS chave,
       COUNT(*) AS chaves_repetidas,
       SUM(qtd - 1) AS linhas_excedentes,
       MAX(qtd) AS pior_caso
  FROM (SELECT `num_reserva`, `num_item_reserva`, `cod_material`, COUNT(*) AS qtd FROM base GROUP BY `num_reserva`, `num_item_reserva`, `cod_material` HAVING COUNT(*) > 1)
ORDER BY chaves_repetidas DESC;

## 6. Varredura de preenchimento — TODAS as colunas

**Seção mais importante do notebook.**

Detecta coluna nunca carregada. Em validação anterior, esta análise revelou 7 colunas
100% nulas no Datalake — uma delas preenchida em **97,9%** dos registros do SAP.
Este erro **não aparece** em teste por amostragem.

Ordene pela coluna `veredito`: os problemas aparecem primeiro.

In [0]:
-- 6. PREENCHIMENTO DE TODAS AS COLUNAS
-- Detecta coluna nunca carregada. Secao mais importante do notebook.
WITH t AS (SELECT COUNT(*) AS total FROM base),
perf AS (
  SELECT stack(36,
    'cod_empresa_compensacao', 'string', COUNT_IF(`cod_empresa_compensacao` IS NULL), COUNT_IF(`cod_empresa_compensacao` IS NOT NULL AND lower(trim(`cod_empresa_compensacao`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_empresa_compensacao`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro', 'string', COUNT_IF(`cod_centro` IS NULL), COUNT_IF(`cod_centro` IS NOT NULL AND lower(trim(`cod_centro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito', 'string', COUNT_IF(`cod_deposito` IS NULL), COUNT_IF(`cod_deposito` IS NOT NULL AND lower(trim(`cod_deposito`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito`) RLIKE '^0+([.,]0+)?$'),
    'num_reserva', 'string', COUNT_IF(`num_reserva` IS NULL), COUNT_IF(`num_reserva` IS NOT NULL AND lower(trim(`num_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_reserva`) RLIKE '^0+([.,]0+)?$'),
    'num_ordem', 'string', COUNT_IF(`num_ordem` IS NULL), COUNT_IF(`num_ordem` IS NOT NULL AND lower(trim(`num_ordem`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_ordem`) RLIKE '^0+([.,]0+)?$'),
    'cod_material', 'string', COUNT_IF(`cod_material` IS NULL), COUNT_IF(`cod_material` IS NOT NULL AND lower(trim(`cod_material`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_material`) RLIKE '^0+([.,]0+)?$'),
    'desc_produto', 'string', COUNT_IF(`desc_produto` IS NULL), COUNT_IF(`desc_produto` IS NOT NULL AND lower(trim(`desc_produto`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`desc_produto`) RLIKE '^0+([.,]0+)?$'),
    'vl_qtd_solicitada', 'decimal(13,3)', COUNT_IF(`vl_qtd_solicitada` IS NULL), 0L, COUNT_IF(`vl_qtd_solicitada` = 0),
    'vl_qtd_retirada', 'decimal(13,3)', COUNT_IF(`vl_qtd_retirada` IS NULL), 0L, COUNT_IF(`vl_qtd_retirada` = 0),
    'sg_unidade_medida_basica', 'string', COUNT_IF(`sg_unidade_medida_basica` IS NULL), COUNT_IF(`sg_unidade_medida_basica` IS NOT NULL AND lower(trim(`sg_unidade_medida_basica`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`sg_unidade_medida_basica`) RLIKE '^0+([.,]0+)?$'),
    'qt_estoque_livre_avaliado', 'decimal(13,3)', COUNT_IF(`qt_estoque_livre_avaliado` IS NULL), 0L, COUNT_IF(`qt_estoque_livre_avaliado` = 0),
    'num_item_reserva', 'string', COUNT_IF(`num_item_reserva` IS NULL), COUNT_IF(`num_item_reserva` IS NOT NULL AND lower(trim(`num_item_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_item_reserva`) RLIKE '^0+([.,]0+)?$'),
    'nm_usuario', 'string', COUNT_IF(`nm_usuario` IS NULL), COUNT_IF(`nm_usuario` IS NOT NULL AND lower(trim(`nm_usuario`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_usuario`) RLIKE '^0+([.,]0+)?$'),
    'dt_reserva', 'string', COUNT_IF(`dt_reserva` IS NULL), COUNT_IF(`dt_reserva` IS NOT NULL AND lower(trim(`dt_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`dt_reserva`) RLIKE '^0+([.,]0+)?$'),
    'cod_tipo_movimento', 'string', COUNT_IF(`cod_tipo_movimento` IS NULL), COUNT_IF(`cod_tipo_movimento` IS NOT NULL AND lower(trim(`cod_tipo_movimento`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_tipo_movimento`) RLIKE '^0+([.,]0+)?$'),
    'dt_necessidade', 'date', COUNT_IF(`dt_necessidade` IS NULL), 0L, 0L,
    'num_lote', 'string', COUNT_IF(`num_lote` IS NULL), COUNT_IF(`num_lote` IS NOT NULL AND lower(trim(`num_lote`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_lote`) RLIKE '^0+([.,]0+)?$'),
    'qt_estoque_consignado', 'decimal(23,3)', COUNT_IF(`qt_estoque_consignado` IS NULL), 0L, COUNT_IF(`qt_estoque_consignado` = 0),
    'cod_fornecedor', 'string', COUNT_IF(`cod_fornecedor` IS NULL), COUNT_IF(`cod_fornecedor` IS NOT NULL AND lower(trim(`cod_fornecedor`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_fornecedor`) RLIKE '^0+([.,]0+)?$'),
    'nm_ponto_descarga', 'string', COUNT_IF(`nm_ponto_descarga` IS NULL), COUNT_IF(`nm_ponto_descarga` IS NOT NULL AND lower(trim(`nm_ponto_descarga`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_ponto_descarga`) RLIKE '^0+([.,]0+)?$'),
    'nm_recebedor_mercadoria', 'string', COUNT_IF(`nm_recebedor_mercadoria` IS NULL), COUNT_IF(`nm_recebedor_mercadoria` IS NOT NULL AND lower(trim(`nm_recebedor_mercadoria`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`nm_recebedor_mercadoria`) RLIKE '^0+([.,]0+)?$'),
    'ind_registro_final', 'string', COUNT_IF(`ind_registro_final` IS NULL), COUNT_IF(`ind_registro_final` IS NOT NULL AND lower(trim(`ind_registro_final`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_registro_final`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_eliminado', 'string', COUNT_IF(`ind_item_eliminado` IS NULL), COUNT_IF(`ind_item_eliminado` IS NOT NULL AND lower(trim(`ind_item_eliminado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_eliminado`) RLIKE '^0+([.,]0+)?$'),
    'ind_movimento_permitido', 'string', COUNT_IF(`ind_movimento_permitido` IS NULL), COUNT_IF(`ind_movimento_permitido` IS NOT NULL AND lower(trim(`ind_movimento_permitido`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_movimento_permitido`) RLIKE '^0+([.,]0+)?$'),
    'num_imobilizado', 'string', COUNT_IF(`num_imobilizado` IS NULL), COUNT_IF(`num_imobilizado` IS NOT NULL AND lower(trim(`num_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'ind_item_dummy', 'string', COUNT_IF(`ind_item_dummy` IS NULL), COUNT_IF(`ind_item_dummy` IS NOT NULL AND lower(trim(`ind_item_dummy`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_item_dummy`) RLIKE '^0+([.,]0+)?$'),
    'ind_material_granel', 'string', COUNT_IF(`ind_material_granel` IS NULL), COUNT_IF(`ind_material_granel` IS NOT NULL AND lower(trim(`ind_material_granel`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_material_granel`) RLIKE '^0+([.,]0+)?$'),
    'num_sub_imobilizado', 'string', COUNT_IF(`num_sub_imobilizado` IS NULL), COUNT_IF(`num_sub_imobilizado` IS NOT NULL AND lower(trim(`num_sub_imobilizado`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_sub_imobilizado`) RLIKE '^0+([.,]0+)?$'),
    'st_reserva', 'string', COUNT_IF(`st_reserva` IS NULL), COUNT_IF(`st_reserva` IS NOT NULL AND lower(trim(`st_reserva`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`st_reserva`) RLIKE '^0+([.,]0+)?$'),
    'tp_registro', 'string', COUNT_IF(`tp_registro` IS NULL), COUNT_IF(`tp_registro` IS NOT NULL AND lower(trim(`tp_registro`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`tp_registro`) RLIKE '^0+([.,]0+)?$'),
    'cod_centro_custo', 'string', COUNT_IF(`cod_centro_custo` IS NULL), COUNT_IF(`cod_centro_custo` IS NOT NULL AND lower(trim(`cod_centro_custo`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_centro_custo`) RLIKE '^0+([.,]0+)?$'),
    'cod_deposito_destino', 'string', COUNT_IF(`cod_deposito_destino` IS NULL), COUNT_IF(`cod_deposito_destino` IS NOT NULL AND lower(trim(`cod_deposito_destino`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_deposito_destino`) RLIKE '^0+([.,]0+)?$'),
    'cod_diagrama_rede', 'string', COUNT_IF(`cod_diagrama_rede` IS NULL), COUNT_IF(`cod_diagrama_rede` IS NOT NULL AND lower(trim(`cod_diagrama_rede`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`cod_diagrama_rede`) RLIKE '^0+([.,]0+)?$'),
    'num_divisao_programa_venda', 'string', COUNT_IF(`num_divisao_programa_venda` IS NULL), COUNT_IF(`num_divisao_programa_venda` IS NOT NULL AND lower(trim(`num_divisao_programa_venda`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`num_divisao_programa_venda`) RLIKE '^0+([.,]0+)?$'),
    'ind_necessidade_atendida', 'string', COUNT_IF(`ind_necessidade_atendida` IS NULL), COUNT_IF(`ind_necessidade_atendida` IS NOT NULL AND lower(trim(`ind_necessidade_atendida`)) IN ('', 'null', 'nan', 'none', '#', '-', 'na', 'n/a')), COUNT_IF(trim(`ind_necessidade_atendida`) RLIKE '^0+([.,]0+)?$'),
    'qt_estoque_projeto', 'decimal(23,3)', COUNT_IF(`qt_estoque_projeto` IS NULL), 0L, COUNT_IF(`qt_estoque_projeto` = 0)
  ) AS (coluna, tipo, nulos, vazios, zeros)
  FROM base
)
SELECT
  p.coluna,
  p.tipo,
  p.nulos,
  p.vazios,
  p.zeros,
  t.total - p.nulos - p.vazios - p.zeros                              AS uteis,
  ROUND(100.0 * (t.total - p.nulos - p.vazios - p.zeros) / t.total, 2) AS pct_util,
  CASE
    WHEN p.nulos = t.total                                      THEN '1. 100% NULO'
    WHEN t.total - p.nulos - p.vazios - p.zeros <= 0            THEN '2. SEM VALOR UTIL'
    WHEN (t.total - p.nulos - p.vazios - p.zeros) < t.total*0.01 THEN '3. QUASE VAZIO (<1%)'
    ELSE '9. ok'
  END AS veredito
FROM perf p CROSS JOIN t
ORDER BY veredito, pct_util, coluna;

## 7. Cardinalidade

Valores distintos por coluna. Coluna constante é candidata a default de carga.

In [0]:
-- 7. CARDINALIDADE
WITH t AS (SELECT COUNT(*) AS total FROM base),
card AS (
  SELECT stack(36,
    'cod_empresa_compensacao', 'string', approx_count_distinct(`cod_empresa_compensacao`),
    'cod_centro', 'string', approx_count_distinct(`cod_centro`),
    'cod_deposito', 'string', approx_count_distinct(`cod_deposito`),
    'num_reserva', 'string', approx_count_distinct(`num_reserva`),
    'num_ordem', 'string', approx_count_distinct(`num_ordem`),
    'cod_material', 'string', approx_count_distinct(`cod_material`),
    'desc_produto', 'string', approx_count_distinct(`desc_produto`),
    'vl_qtd_solicitada', 'decimal(13,3)', approx_count_distinct(`vl_qtd_solicitada`),
    'vl_qtd_retirada', 'decimal(13,3)', approx_count_distinct(`vl_qtd_retirada`),
    'sg_unidade_medida_basica', 'string', approx_count_distinct(`sg_unidade_medida_basica`),
    'qt_estoque_livre_avaliado', 'decimal(13,3)', approx_count_distinct(`qt_estoque_livre_avaliado`),
    'num_item_reserva', 'string', approx_count_distinct(`num_item_reserva`),
    'nm_usuario', 'string', approx_count_distinct(`nm_usuario`),
    'dt_reserva', 'string', approx_count_distinct(`dt_reserva`),
    'cod_tipo_movimento', 'string', approx_count_distinct(`cod_tipo_movimento`),
    'dt_necessidade', 'date', approx_count_distinct(`dt_necessidade`),
    'num_lote', 'string', approx_count_distinct(`num_lote`),
    'qt_estoque_consignado', 'decimal(23,3)', approx_count_distinct(`qt_estoque_consignado`),
    'cod_fornecedor', 'string', approx_count_distinct(`cod_fornecedor`),
    'nm_ponto_descarga', 'string', approx_count_distinct(`nm_ponto_descarga`),
    'nm_recebedor_mercadoria', 'string', approx_count_distinct(`nm_recebedor_mercadoria`),
    'ind_registro_final', 'string', approx_count_distinct(`ind_registro_final`),
    'ind_item_eliminado', 'string', approx_count_distinct(`ind_item_eliminado`),
    'ind_movimento_permitido', 'string', approx_count_distinct(`ind_movimento_permitido`),
    'num_imobilizado', 'string', approx_count_distinct(`num_imobilizado`),
    'ind_item_dummy', 'string', approx_count_distinct(`ind_item_dummy`),
    'ind_material_granel', 'string', approx_count_distinct(`ind_material_granel`),
    'num_sub_imobilizado', 'string', approx_count_distinct(`num_sub_imobilizado`),
    'st_reserva', 'string', approx_count_distinct(`st_reserva`),
    'tp_registro', 'string', approx_count_distinct(`tp_registro`),
    'cod_centro_custo', 'string', approx_count_distinct(`cod_centro_custo`),
    'cod_deposito_destino', 'string', approx_count_distinct(`cod_deposito_destino`),
    'cod_diagrama_rede', 'string', approx_count_distinct(`cod_diagrama_rede`),
    'num_divisao_programa_venda', 'string', approx_count_distinct(`num_divisao_programa_venda`),
    'ind_necessidade_atendida', 'string', approx_count_distinct(`ind_necessidade_atendida`),
    'qt_estoque_projeto', 'decimal(23,3)', approx_count_distinct(`qt_estoque_projeto`)
  ) AS (coluna, tipo, distintos)
  FROM base
)
SELECT c.coluna, c.tipo, c.distintos,
       ROUND(100.0 * c.distintos / t.total, 4) AS pct_distintos,
       CASE
         WHEN c.distintos <= 1                   THEN '1. CONSTANTE (1 valor)'
         WHEN c.distintos <= 3                   THEN '2. cardinalidade muito baixa'
         WHEN c.distintos > t.total * 0.95       THEN '3. candidata a identificador'
         ELSE '9. normal'
       END AS classificacao
FROM card c CROSS JOIN t
ORDER BY c.distintos;

## 8. Domínio das colunas categóricas

Top 8 valores de cada coluna. Um valor concentrando mais de 99% da base
indica possível default de carga em vez de dado real.

In [0]:
-- 8. DOMINIO DAS COLUNAS CATEGORICAS (top 8 de cada)
-- Valor concentrando >99% indica possivel default de carga.
(SELECT 'cod_empresa_compensacao' AS coluna, CAST(`cod_empresa_compensacao` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_empresa_compensacao` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_centro' AS coluna, CAST(`cod_centro` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_centro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_deposito' AS coluna, CAST(`cod_deposito` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_deposito` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'cod_tipo_movimento' AS coluna, CAST(`cod_tipo_movimento` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `cod_tipo_movimento` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'st_reserva' AS coluna, CAST(`st_reserva` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `st_reserva` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'tp_registro' AS coluna, CAST(`tp_registro` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `tp_registro` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_registro_final' AS coluna, CAST(`ind_registro_final` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_registro_final` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_eliminado' AS coluna, CAST(`ind_item_eliminado` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_item_eliminado` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_movimento_permitido' AS coluna, CAST(`ind_movimento_permitido` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_movimento_permitido` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_item_dummy' AS coluna, CAST(`ind_item_dummy` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_item_dummy` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_material_granel' AS coluna, CAST(`ind_material_granel` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_material_granel` ORDER BY qtd DESC LIMIT 8)
UNION ALL
(SELECT 'ind_necessidade_atendida' AS coluna, CAST(`ind_necessidade_atendida` AS STRING) AS valor,
        COUNT(*) AS qtd,
        ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
   FROM base GROUP BY `ind_necessidade_atendida` ORDER BY qtd DESC LIMIT 8)
ORDER BY coluna, qtd DESC;

## 9. Perfil dos campos numéricos

**Atenção ao tipo:** quantidade costuma usar `decimal`, mas valor monetário
frequentemente usa `double` — risco de arredondamento na conciliação financeira.

In [0]:
-- 9. PERFIL DOS CAMPOS NUMERICOS
-- Tipo DOUBLE em valor monetario = risco de arredondamento na conciliacao.
SELECT * FROM (
  SELECT stack(5,
    'vl_qtd_solicitada', 'decimal(13,3)', COUNT(`vl_qtd_solicitada`), CAST(MIN(`vl_qtd_solicitada`) AS DOUBLE), CAST(MAX(`vl_qtd_solicitada`) AS DOUBLE), CAST(AVG(`vl_qtd_solicitada`) AS DOUBLE), CAST(percentile_approx(`vl_qtd_solicitada`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_qtd_solicitada`, 0.95) AS DOUBLE), COUNT_IF(`vl_qtd_solicitada` < 0), COUNT_IF(`vl_qtd_solicitada` = 0),
    'vl_qtd_retirada', 'decimal(13,3)', COUNT(`vl_qtd_retirada`), CAST(MIN(`vl_qtd_retirada`) AS DOUBLE), CAST(MAX(`vl_qtd_retirada`) AS DOUBLE), CAST(AVG(`vl_qtd_retirada`) AS DOUBLE), CAST(percentile_approx(`vl_qtd_retirada`, 0.5) AS DOUBLE), CAST(percentile_approx(`vl_qtd_retirada`, 0.95) AS DOUBLE), COUNT_IF(`vl_qtd_retirada` < 0), COUNT_IF(`vl_qtd_retirada` = 0),
    'qt_estoque_livre_avaliado', 'decimal(13,3)', COUNT(`qt_estoque_livre_avaliado`), CAST(MIN(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(MAX(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(AVG(`qt_estoque_livre_avaliado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_livre_avaliado`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_estoque_livre_avaliado`, 0.95) AS DOUBLE), COUNT_IF(`qt_estoque_livre_avaliado` < 0), COUNT_IF(`qt_estoque_livre_avaliado` = 0),
    'qt_estoque_consignado', 'decimal(23,3)', COUNT(`qt_estoque_consignado`), CAST(MIN(`qt_estoque_consignado`) AS DOUBLE), CAST(MAX(`qt_estoque_consignado`) AS DOUBLE), CAST(AVG(`qt_estoque_consignado`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_consignado`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_estoque_consignado`, 0.95) AS DOUBLE), COUNT_IF(`qt_estoque_consignado` < 0), COUNT_IF(`qt_estoque_consignado` = 0),
    'qt_estoque_projeto', 'decimal(23,3)', COUNT(`qt_estoque_projeto`), CAST(MIN(`qt_estoque_projeto`) AS DOUBLE), CAST(MAX(`qt_estoque_projeto`) AS DOUBLE), CAST(AVG(`qt_estoque_projeto`) AS DOUBLE), CAST(percentile_approx(`qt_estoque_projeto`, 0.5) AS DOUBLE), CAST(percentile_approx(`qt_estoque_projeto`, 0.95) AS DOUBLE), COUNT_IF(`qt_estoque_projeto` < 0), COUNT_IF(`qt_estoque_projeto` = 0)
  ) AS (coluna, tipo, preenchidos, minimo, maximo, media, mediana, p95, negativos, zeros)
  FROM base
)
ORDER BY coluna;

## 10. Datas armazenadas como STRING

**Armadilha conhecida:** o SAP exporta `2024-02-23 00:00:00` e o Datalake grava `20240223`.
É a mesma data em formato diferente — já gerou **16.773 falsos positivos**.

Se a coluna `veredito` acusar mais de um formato, a normalização é obrigatória.

In [0]:
-- 10. DATAS ARMAZENADAS COMO STRING
-- ARMADILHA: SAP exporta '2024-02-23 00:00:00', Datalake grava '20240223'.
-- Mesma data, formato diferente. Ja gerou 16.773 falsos positivos.
WITH t AS (SELECT COUNT(*) AS total FROM base),
d AS (
  SELECT stack(1,
    'dt_reserva', COUNT_IF(`dt_reserva` IS NULL OR trim(`dt_reserva`) = ''), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{8}$'), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{4}-[0-9]{2}-[0-9]{2}'), COUNT_IF(trim(`dt_reserva`) RLIKE '^[0-9]{2}/[0-9]{2}/[0-9]{4}'), COUNT_IF(trim(`dt_reserva`) IN ('00000000', '0000-00-00', '0')), MIN(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END), MAX(CASE WHEN trim(`dt_reserva`) NOT IN ('', '00000000') THEN `dt_reserva` END)
  ) AS (coluna, vazios, fmt_AAAAMMDD, fmt_ISO, fmt_BR, data_zero, minimo, maximo)
  FROM base
)
SELECT d.coluna, d.vazios, d.fmt_AAAAMMDD, d.fmt_ISO, d.fmt_BR, d.data_zero,
       t.total - d.vazios - d.fmt_AAAAMMDD - d.fmt_ISO - d.fmt_BR AS nao_reconhecido,
       d.minimo, d.maximo,
       CASE WHEN (CASE WHEN d.fmt_AAAAMMDD > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_ISO       > 0 THEN 1 ELSE 0 END
                + CASE WHEN d.fmt_BR        > 0 THEN 1 ELSE 0 END) > 1
            THEN 'ALERTA: mais de um formato na mesma coluna'
            ELSE 'formato unico' END AS veredito
FROM d CROSS JOIN t
ORDER BY d.coluna;

## 10.1 Datas em tipo nativo

In [0]:
-- 10.1 DATAS EM TIPO NATIVO
SELECT * FROM (
  SELECT stack(1,
    'dt_necessidade', COUNT_IF(`dt_necessidade` IS NULL), CAST(MIN(`dt_necessidade`) AS STRING), CAST(MAX(`dt_necessidade`) AS STRING), COUNT(DISTINCT `dt_necessidade`), COUNT_IF(`dt_necessidade` > current_date()), COUNT_IF(`dt_necessidade` < DATE'1990-01-01')
  ) AS (coluna, nulos, minimo, maximo, distintas, futuras, anteriores_1990)
  FROM base
)
ORDER BY coluna;

## 11. Códigos — zeros à esquerda, espaços e formato

**Armadilha conhecida:** o SAP exporta `425263` e o Datalake grava `000000000000425263`.
Sem normalizar, o join dá **0% de match**.

A coluna `alertas` resume o que exige tratamento antes da comparação.

In [0]:
-- 11. CODIGOS: ZEROS A ESQUERDA, ESPACOS E FORMATO
-- ARMADILHA: SAP grava '425263', Datalake grava '000000000000425263'.
-- Sem normalizar, o join da 0% de match.
SELECT coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
       distintos_bruto, distintos_sem_zeros,
       distintos_bruto - distintos_sem_zeros AS colisoes_ao_remover_zeros,
       CONCAT_WS(' | ',
         CASE WHEN com_zeros_esq > 0 THEN 'tem zeros a esquerda' END,
         CASE WHEN len_min <> len_max THEN 'comprimento variavel' END,
         CASE WHEN com_espacos > 0 THEN 'tem espacos' END,
         CASE WHEN distintos_bruto - distintos_sem_zeros > 0 THEN 'COLISAO ao remover zeros' END,
         CASE WHEN tipo LIKE '%int%' OR tipo LIKE 'big%'
              THEN 'TIPO NUMERICO - zeros a esquerda JA perdidos' END
       ) AS alertas
FROM (
  SELECT stack(7,
    'num_reserva', 'string', COUNT_IF(CAST(`num_reserva` AS STRING) IS NULL OR trim(CAST(`num_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_reserva` AS STRING)))), MAX(length(trim(CAST(`num_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_reserva` AS STRING) <> trim(CAST(`num_reserva` AS STRING))), COUNT(DISTINCT trim(CAST(`num_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_reserva` AS STRING)), '^0+', '')),
    'num_item_reserva', 'string', COUNT_IF(CAST(`num_item_reserva` AS STRING) IS NULL OR trim(CAST(`num_item_reserva` AS STRING)) = ''), MIN(length(trim(CAST(`num_item_reserva` AS STRING)))), MAX(length(trim(CAST(`num_item_reserva` AS STRING)))), COUNT_IF(trim(CAST(`num_item_reserva` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_item_reserva` AS STRING) <> trim(CAST(`num_item_reserva` AS STRING))), COUNT(DISTINCT trim(CAST(`num_item_reserva` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_item_reserva` AS STRING)), '^0+', '')),
    'cod_material', 'string', COUNT_IF(CAST(`cod_material` AS STRING) IS NULL OR trim(CAST(`cod_material` AS STRING)) = ''), MIN(length(trim(CAST(`cod_material` AS STRING)))), MAX(length(trim(CAST(`cod_material` AS STRING)))), COUNT_IF(trim(CAST(`cod_material` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_material` AS STRING) <> trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_material` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '')),
    'num_ordem', 'string', COUNT_IF(CAST(`num_ordem` AS STRING) IS NULL OR trim(CAST(`num_ordem` AS STRING)) = ''), MIN(length(trim(CAST(`num_ordem` AS STRING)))), MAX(length(trim(CAST(`num_ordem` AS STRING)))), COUNT_IF(trim(CAST(`num_ordem` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_ordem` AS STRING) <> trim(CAST(`num_ordem` AS STRING))), COUNT(DISTINCT trim(CAST(`num_ordem` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_ordem` AS STRING)), '^0+', '')),
    'cod_fornecedor', 'string', COUNT_IF(CAST(`cod_fornecedor` AS STRING) IS NULL OR trim(CAST(`cod_fornecedor` AS STRING)) = ''), MIN(length(trim(CAST(`cod_fornecedor` AS STRING)))), MAX(length(trim(CAST(`cod_fornecedor` AS STRING)))), COUNT_IF(trim(CAST(`cod_fornecedor` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_fornecedor` AS STRING) <> trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_fornecedor` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_fornecedor` AS STRING)), '^0+', '')),
    'cod_centro_custo', 'string', COUNT_IF(CAST(`cod_centro_custo` AS STRING) IS NULL OR trim(CAST(`cod_centro_custo` AS STRING)) = ''), MIN(length(trim(CAST(`cod_centro_custo` AS STRING)))), MAX(length(trim(CAST(`cod_centro_custo` AS STRING)))), COUNT_IF(trim(CAST(`cod_centro_custo` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`cod_centro_custo` AS STRING) <> trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT trim(CAST(`cod_centro_custo` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`cod_centro_custo` AS STRING)), '^0+', '')),
    'num_imobilizado', 'string', COUNT_IF(CAST(`num_imobilizado` AS STRING) IS NULL OR trim(CAST(`num_imobilizado` AS STRING)) = ''), MIN(length(trim(CAST(`num_imobilizado` AS STRING)))), MAX(length(trim(CAST(`num_imobilizado` AS STRING)))), COUNT_IF(trim(CAST(`num_imobilizado` AS STRING)) RLIKE '^0[0-9]'), COUNT_IF(CAST(`num_imobilizado` AS STRING) <> trim(CAST(`num_imobilizado` AS STRING))), COUNT(DISTINCT trim(CAST(`num_imobilizado` AS STRING))), COUNT(DISTINCT regexp_replace(trim(CAST(`num_imobilizado` AS STRING)), '^0+', ''))
  ) AS (coluna, tipo, vazios, len_min, len_max, com_zeros_esq, com_espacos,
        distintos_bruto, distintos_sem_zeros)
  FROM base
)
ORDER BY coluna;

## 12. Amostra de linhas completas

O dado como ele realmente está: formato de código, decimais, datas e nulos.

In [0]:
-- 12. AMOSTRA
SELECT * FROM base LIMIT 20;

In [0]:
-- 12.1 AMOSTRA ALEATORIA
SELECT * FROM base ORDER BY rand() LIMIT 10;

## 13. Distribuição por dimensão de recorte

Base para escolher o cenário de teste: volume viável (10 mil a 300 mil linhas)
contendo os casos-limite identificados nas seções anteriores.

In [0]:
-- 13. DISTRIBUICAO POR cod_centro
SELECT `cod_centro`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_centro`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_empresa_compensacao
SELECT `cod_empresa_compensacao`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_empresa_compensacao`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR cod_tipo_movimento
SELECT `cod_tipo_movimento`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `cod_tipo_movimento`
ORDER BY linhas DESC
LIMIT 40;

In [0]:
-- 13. DISTRIBUICAO POR st_reserva
SELECT `st_reserva`,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct,
       CASE WHEN COUNT(*) BETWEEN 10000 AND 300000
            THEN 'CANDIDATO A CENARIO DE TESTE' ELSE '' END AS sugestao
FROM base
GROUP BY `st_reserva`
ORDER BY linhas DESC
LIMIT 40;

## 14. Duplicidade — o que diferencia as linhas repetidas?

Analisando pela chave **num_reserva + cod_material**.

**Regra crítica:** linhas **idênticas** = duplicata real (erro de carga).
Linhas **distintas** = granularidade adicional legítima (split valuation, lote, tipo de avaliação).

São problemas diferentes com tratamentos diferentes. Em validação anterior, 5 linhas do mesmo
material eram todas distintas, diferenciadas por um campo que sequer existia nos extratos do SAP.

In [0]:
-- 14. CHAVES DUPLICADAS
SELECT `num_reserva`, `cod_material`, COUNT(*) AS qtd
FROM base
GROUP BY `num_reserva`, `cod_material`
HAVING COUNT(*) > 1
ORDER BY qtd DESC
LIMIT 20;

In [0]:
-- 14.1 O QUE DIFERENCIA AS LINHAS DUPLICADAS?
-- REGRA: linhas identicas = duplicata real (erro de carga).
--        linhas distintas = granularidade adicional legitima (split valuation, lote...).
WITH dup AS (
  SELECT `num_reserva`, `cod_material` FROM base GROUP BY `num_reserva`, `cod_material` HAVING COUNT(*) > 1
),
d AS (
  SELECT b.* FROM base b JOIN dup USING (`num_reserva`, `cod_material`)
),
agg AS (
  SELECT `num_reserva`, `cod_material`,
         COUNT(DISTINCT `cod_empresa_compensacao`) AS `cod_empresa_compensacao`,
         COUNT(DISTINCT `cod_centro`) AS `cod_centro`,
         COUNT(DISTINCT `cod_deposito`) AS `cod_deposito`,
         COUNT(DISTINCT `num_ordem`) AS `num_ordem`,
         COUNT(DISTINCT `desc_produto`) AS `desc_produto`,
         COUNT(DISTINCT `vl_qtd_solicitada`) AS `vl_qtd_solicitada`,
         COUNT(DISTINCT `vl_qtd_retirada`) AS `vl_qtd_retirada`,
         COUNT(DISTINCT `sg_unidade_medida_basica`) AS `sg_unidade_medida_basica`,
         COUNT(DISTINCT `qt_estoque_livre_avaliado`) AS `qt_estoque_livre_avaliado`,
         COUNT(DISTINCT `num_item_reserva`) AS `num_item_reserva`,
         COUNT(DISTINCT `nm_usuario`) AS `nm_usuario`,
         COUNT(DISTINCT `dt_reserva`) AS `dt_reserva`,
         COUNT(DISTINCT `cod_tipo_movimento`) AS `cod_tipo_movimento`,
         COUNT(DISTINCT `dt_necessidade`) AS `dt_necessidade`,
         COUNT(DISTINCT `num_lote`) AS `num_lote`,
         COUNT(DISTINCT `qt_estoque_consignado`) AS `qt_estoque_consignado`,
         COUNT(DISTINCT `cod_fornecedor`) AS `cod_fornecedor`,
         COUNT(DISTINCT `nm_ponto_descarga`) AS `nm_ponto_descarga`,
         COUNT(DISTINCT `nm_recebedor_mercadoria`) AS `nm_recebedor_mercadoria`,
         COUNT(DISTINCT `ind_registro_final`) AS `ind_registro_final`,
         COUNT(DISTINCT `ind_item_eliminado`) AS `ind_item_eliminado`,
         COUNT(DISTINCT `ind_movimento_permitido`) AS `ind_movimento_permitido`,
         COUNT(DISTINCT `num_imobilizado`) AS `num_imobilizado`,
         COUNT(DISTINCT `ind_item_dummy`) AS `ind_item_dummy`,
         COUNT(DISTINCT `ind_material_granel`) AS `ind_material_granel`,
         COUNT(DISTINCT `num_sub_imobilizado`) AS `num_sub_imobilizado`,
         COUNT(DISTINCT `st_reserva`) AS `st_reserva`,
         COUNT(DISTINCT `tp_registro`) AS `tp_registro`,
         COUNT(DISTINCT `cod_centro_custo`) AS `cod_centro_custo`,
         COUNT(DISTINCT `cod_deposito_destino`) AS `cod_deposito_destino`,
         COUNT(DISTINCT `cod_diagrama_rede`) AS `cod_diagrama_rede`,
         COUNT(DISTINCT `num_divisao_programa_venda`) AS `num_divisao_programa_venda`,
         COUNT(DISTINCT `ind_necessidade_atendida`) AS `ind_necessidade_atendida`,
         COUNT(DISTINCT `qt_estoque_projeto`) AS `qt_estoque_projeto`
  FROM d GROUP BY `num_reserva`, `cod_material`
)
SELECT coluna, max_valores_distintos,
       CASE WHEN max_valores_distintos > 1
            THEN 'VARIA - faz parte da chave real'
            ELSE 'constante' END AS veredito
FROM (
  SELECT stack(34,
    'cod_empresa_compensacao', MAX(`cod_empresa_compensacao`),
    'cod_centro', MAX(`cod_centro`),
    'cod_deposito', MAX(`cod_deposito`),
    'num_ordem', MAX(`num_ordem`),
    'desc_produto', MAX(`desc_produto`),
    'vl_qtd_solicitada', MAX(`vl_qtd_solicitada`),
    'vl_qtd_retirada', MAX(`vl_qtd_retirada`),
    'sg_unidade_medida_basica', MAX(`sg_unidade_medida_basica`),
    'qt_estoque_livre_avaliado', MAX(`qt_estoque_livre_avaliado`),
    'num_item_reserva', MAX(`num_item_reserva`),
    'nm_usuario', MAX(`nm_usuario`),
    'dt_reserva', MAX(`dt_reserva`),
    'cod_tipo_movimento', MAX(`cod_tipo_movimento`),
    'dt_necessidade', MAX(`dt_necessidade`),
    'num_lote', MAX(`num_lote`),
    'qt_estoque_consignado', MAX(`qt_estoque_consignado`),
    'cod_fornecedor', MAX(`cod_fornecedor`),
    'nm_ponto_descarga', MAX(`nm_ponto_descarga`),
    'nm_recebedor_mercadoria', MAX(`nm_recebedor_mercadoria`),
    'ind_registro_final', MAX(`ind_registro_final`),
    'ind_item_eliminado', MAX(`ind_item_eliminado`),
    'ind_movimento_permitido', MAX(`ind_movimento_permitido`),
    'num_imobilizado', MAX(`num_imobilizado`),
    'ind_item_dummy', MAX(`ind_item_dummy`),
    'ind_material_granel', MAX(`ind_material_granel`),
    'num_sub_imobilizado', MAX(`num_sub_imobilizado`),
    'st_reserva', MAX(`st_reserva`),
    'tp_registro', MAX(`tp_registro`),
    'cod_centro_custo', MAX(`cod_centro_custo`),
    'cod_deposito_destino', MAX(`cod_deposito_destino`),
    'cod_diagrama_rede', MAX(`cod_diagrama_rede`),
    'num_divisao_programa_venda', MAX(`num_divisao_programa_venda`),
    'ind_necessidade_atendida', MAX(`ind_necessidade_atendida`),
    'qt_estoque_projeto', MAX(`qt_estoque_projeto`)
  ) AS (coluna, max_valores_distintos)
  FROM agg
)
ORDER BY max_valores_distintos DESC, coluna;

## 15. Freshness — atualidade da carga

In [0]:
-- 15. FRESHNESS
-- Esta tabela NAO possui coluna de data de ingestao.
-- ACAO: solicitar ao time de dados a inclusao de dateingest ou equivalente.
DESCRIBE HISTORY dev_procurement.corp_curated.tbl_ds_pro_zglmm368 LIMIT 10;

## 16. Análises específicas — ZGLMM368

### 16.1 A chave inclui `num_item_reserva`?

O clustering declarado é **reserva + material**, mas existe `num_item_reserva`.
Uma reserva pode ter vários itens, e o mesmo material pode aparecer em mais de um item.

In [0]:
-- 16.1 ITENS POR RESERVA
SELECT itens_na_reserva, COUNT(*) AS reservas
FROM (SELECT num_reserva, COUNT(DISTINCT num_item_reserva) AS itens_na_reserva
        FROM base GROUP BY num_reserva)
GROUP BY itens_na_reserva
ORDER BY itens_na_reserva
LIMIT 30;

In [0]:
-- 16.1b MATERIAL REPETIDO NA MESMA RESERVA
SELECT num_reserva, cod_material,
       COUNT(DISTINCT num_item_reserva) AS itens
FROM base
GROUP BY num_reserva, cod_material
HAVING COUNT(DISTINCT num_item_reserva) > 1
ORDER BY itens DESC
LIMIT 20;

### 16.2 Quantidade solicitada × retirada
`vl_qtd_retirada` não deveria exceder `vl_qtd_solicitada`.

In [0]:
-- 16.2 SOLICITADO x RETIRADO
SELECT COUNT(*) AS linhas,
       COUNT_IF(vl_qtd_retirada > vl_qtd_solicitada) AS retirada_maior_que_solicitada,
       COUNT_IF(vl_qtd_retirada = vl_qtd_solicitada) AS totalmente_atendida,
       COUNT_IF(vl_qtd_retirada = 0) AS nada_retirado,
       COUNT_IF(vl_qtd_solicitada = 0) AS solicitada_zero,
       COUNT_IF(vl_qtd_solicitada < 0) AS solicitada_negativa,
       COUNT_IF(vl_qtd_retirada < 0) AS retirada_negativa,
       CASE WHEN COUNT_IF(vl_qtd_retirada > vl_qtd_solicitada) > 0
            THEN 'ATENCAO - confirmar se e regra de negocio ou erro' ELSE 'ok' END AS veredito
FROM base;

In [0]:
-- 16.2b STATUS x ATENDIMENTO
SELECT st_reserva,
       CASE WHEN vl_qtd_retirada = 0 THEN 'nada retirado'
            WHEN vl_qtd_retirada >= vl_qtd_solicitada THEN 'total'
            ELSE 'parcial' END AS atendimento,
       COUNT(*) AS linhas
FROM base
GROUP BY st_reserva,
         CASE WHEN vl_qtd_retirada = 0 THEN 'nada retirado'
              WHEN vl_qtd_retirada >= vl_qtd_solicitada THEN 'total'
              ELSE 'parcial' END
ORDER BY linhas DESC
LIMIT 30;

### 16.3 Classificação contábil e vínculo com ordem

In [0]:
-- 16.3 CLASSIFICACAO CONTABIL
SELECT CASE WHEN num_ordem IS NOT NULL AND trim(num_ordem) <> '' THEN 'ordem'
            WHEN cod_centro_custo IS NOT NULL AND trim(cod_centro_custo) <> '' THEN 'centro de custo'
            WHEN num_imobilizado IS NOT NULL AND trim(num_imobilizado) <> '' THEN 'imobilizado'
            ELSE 'sem classificacao' END AS classificacao,
       COUNT(*) AS linhas,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM base), 2) AS pct
FROM base
GROUP BY 1
ORDER BY linhas DESC;

## 90. Integridade referencial cruzada _(opcional)_

Confere se os códigos desta tabela existem nas tabelas de referência.
Execute apenas se as outras tabelas estiverem acessíveis no mesmo ambiente.

In [0]:
-- 90. INTEGRIDADE: cod_material -> dev_procurement.corp_curated.tbl_ds_mdm_mm60.cod_material
WITH loc AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM base WHERE `cod_material` IS NOT NULL AND trim(CAST(`cod_material` AS STRING)) <> ''
),
ref AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_material` AS STRING)), '^0+', '') AS k
  FROM dev_procurement.corp_curated.tbl_ds_mdm_mm60
)
SELECT (SELECT COUNT(*) FROM loc) AS codigos_distintos_aqui,
       (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k)) AS sem_correspondencia,
       ROUND(100.0 * (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k))
                   / (SELECT COUNT(*) FROM loc), 2) AS pct_orfao;

In [0]:
-- 90. INTEGRIDADE: num_ordem -> dev_procurement.corp_curated.tbl_ds_ind_iw39.cod_ordem
WITH loc AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`num_ordem` AS STRING)), '^0+', '') AS k
  FROM base WHERE `num_ordem` IS NOT NULL AND trim(CAST(`num_ordem` AS STRING)) <> ''
),
ref AS (
  SELECT DISTINCT regexp_replace(trim(CAST(`cod_ordem` AS STRING)), '^0+', '') AS k
  FROM dev_procurement.corp_curated.tbl_ds_ind_iw39
)
SELECT (SELECT COUNT(*) FROM loc) AS codigos_distintos_aqui,
       (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k)) AS sem_correspondencia,
       ROUND(100.0 * (SELECT COUNT(*) FROM loc LEFT ANTI JOIN ref USING (k))
                   / (SELECT COUNT(*) FROM loc), 2) AS pct_orfao;

## 99. Resumo consolidado

**Copie a saída desta célula** para o relatório ou para a base de conhecimento do agente.

In [0]:
-- 99. RESUMO CONSOLIDADO
SELECT 'VOLUMETRIA' AS bloco, 'linhas na base' AS item,
       CAST(COUNT(*) AS STRING) AS valor, '' AS veredito
  FROM base
UNION ALL
SELECT 'VOLUMETRIA', 'colunas', '36', ''
UNION ALL
SELECT 'VOLUMETRIA', 'clustering declarado',
       'num_reserva, cod_material', ''
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_reserva' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_reserva` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_reserva + cod_material' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_reserva`, `cod_material` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_reserva + num_item_reserva' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva` FROM base)
UNION ALL
SELECT 'GRANULARIDADE' AS bloco, 'num_reserva + num_item_reserva + cod_material' AS item,
       CAST(ROUND((SELECT COUNT(*) FROM base) / COUNT(*), 4) AS STRING) AS valor,
       CASE WHEN COUNT(*) = (SELECT COUNT(*) FROM base)
            THEN 'CHAVE UNICA' ELSE 'nao unica' END AS veredito
  FROM (SELECT DISTINCT `num_reserva`, `num_item_reserva`, `cod_material` FROM base)
ORDER BY bloco, item;

---

## Próximo passo

1. Escolher o recorte de teste com base na **seção 13**.
2. Extrair a transação no SAP com o **mesmo recorte** e na **mesma data** do snapshot.
3. Extrair **todas** as abas/telas da transação — comparar parcialmente esconde erros de granularidade.
4. Submeter os arquivos ao agente de validação junto com este notebook executado.

### Checklist antes de comparar com o SAP

- [ ] Chave real identificada (seção 4)
- [ ] Colunas 100% nulas conferidas no SAP antes de classificar como erro (seção 6)
- [ ] Zeros à esquerda normalizados nos dois lados (seção 11)
- [ ] Formato de data normalizado para `AAAAMMDD` (seção 10)
- [ ] Tolerância de 0,005 aplicada em campos `double` (seção 9)
- [ ] Duplicidades classificadas: idênticas vs granularidade legítima (seção 14)
